# Notebook 1: Exploration (EDA)
**Goal:** understand the MovieLens 100K data *before* modelling. Every finding here
should shape a modelling decision later (split strategy, popularity bias, cold start).

Sections: load, data quality, ratings, users, movies (long tail), sparsity, genres, time.

In [ ]:
import os
from pathlib import Path

# Run from the project root so `recommender` and the data/ folder are found
if Path.cwd().name == "notebooks":
    os.chdir("..")

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from recommender.config import DATA_DIR, GENRE_COLS, MIN_USER_RATINGS
from recommender.data import load_movielens, validate_data

pd.set_option("display.width", 140)
print("Working directory:", Path.cwd())

## 1. Load the data
`load_movielens` downloads MovieLens 100K into `data/` the first time.

In [ ]:
movies, ratings = load_movielens(DATA_DIR)
display(ratings.head())
display(movies[["movie_id", "title", "release_date"] + GENRE_COLS[:5]].head())
print("ratings:", ratings.shape, "| movies:", movies.shape)
print("users:", ratings.user_id.nunique(), "| movies rated:", ratings.movie_id.nunique())

## 2. Data quality
Production systems validate assumptions *before* training. `validate_data` raises a
`ValueError` if ratings are out of range, duplicated or reference unknown movies.

In [ ]:
validate_data(movies, ratings)
print("validate_data passed")
print("missing values in ratings:", int(ratings.isna().sum().sum()))
print("duplicate (user, movie) pairs:", int(ratings.duplicated(["user_id", "movie_id"]).sum()))
print("rating range:", ratings.rating.min(), "to", ratings.rating.max())

## 3. How do people rate?

In [ ]:
counts = ratings.rating.value_counts().sort_index()
ax = counts.plot(kind="bar", figsize=(6, 3.5), title="Rating distribution")
ax.set_xlabel("rating"); ax.set_ylabel("count")
plt.show()
print("mean rating:", round(ratings.rating.mean(), 3))
print("share of ratings >= 4 ('liked'):", round((ratings.rating >= 4).mean(), 3))

**Write in your own words:** are ratings skewed positive? What does that mean for
treating `rating >= 4` as "liked"?

## 4. Users: how active are they?

In [ ]:
per_user = ratings.groupby("user_id").size()
print(per_user.describe().round(1))
ax = per_user.plot(kind="hist", bins=40, figsize=(6, 3.5), title="Ratings per user")
ax.set_xlabel("number of ratings")
plt.show()
print("users below MIN_USER_RATINGS:", int((per_user < MIN_USER_RATINGS).sum()))

## 5. Movies: the long tail

In [ ]:
per_movie = ratings.groupby("movie_id").size().sort_values(ascending=False)
print(per_movie.describe().round(1))
top10 = per_movie.head(int(len(per_movie) * 0.10)).sum() / per_movie.sum()
print(f"Top 10% of movies receive {top10:.1%} of all ratings")
print("movies with fewer than 5 ratings:", int((per_movie < 5).sum()))
ax = per_movie.reset_index(drop=True).plot(figsize=(6, 3.5), title="Long tail: ratings per movie (ranked)")
ax.set_xlabel("movie rank"); ax.set_ylabel("ratings")
plt.show()

**Why this matters:** a few blockbusters dominate. A model that just recommends
popular movies will look decent on accuracy metrics but has poor coverage and
diversity. This is why we report coverage and popularity bias in evaluation.

## 6. Sparsity

In [ ]:
density = len(ratings) / (ratings.user_id.nunique() * movies.movie_id.nunique())
print(f"Matrix density: {density:.2%} (so {1 - density:.2%} of user-movie pairs are unknown)")

## 7. Genres

In [ ]:
genre_counts = movies[GENRE_COLS].sum().sort_values(ascending=False)
genre_counts.plot(kind="bar", figsize=(9, 3.5), title="Movies per genre")
plt.show()

means = {}
for g in GENRE_COLS:
    ids = movies.loc[movies[g] == 1, "movie_id"]
    sub = ratings[ratings.movie_id.isin(ids)]
    if len(sub):
        means[g] = sub.rating.mean()
pd.Series(means).sort_values().plot(kind="barh", figsize=(6, 5), title="Mean rating by genre")
plt.show()
print("movies with more than one genre:", int((movies[GENRE_COLS].sum(axis=1) > 1).sum()))
print("distinct genre combinations:", movies[GENRE_COLS].drop_duplicates().shape[0])

**Note the last number:** many movies share *identical* genre vectors, so a
genre-only content model will produce many ties. Keep this in mind in Notebook 2.

## 8. Time

In [ ]:
when = pd.to_datetime(ratings.timestamp, unit="s")
print("first rating:", when.min(), "| last rating:", when.max())
when.dt.to_period("M").value_counts().sort_index().plot(figsize=(9, 3), title="Ratings per month")
plt.show()

## 9. Takeaways (fill in)
1. Rating behaviour: ...
2. Long tail / popularity bias: ...
3. Sparsity and what it means for collaborative filtering: ...
4. Genre features and their limits: ...
5. Decisions this drives for the model: chronological split, min-ratings filter, coverage metric ...